# 모델 A 피처 복구와 여러 모델 비교 실험

이 노트북은 **실행하지 않은 템플릿**입니다. 수집·학습·평가·테스트·백그라운드 작업을 자동 시작하지 않습니다.
설정 셀의 `LOAD_DATA`, `RUN_DEV` 등을 사용자가 켠 뒤 원하는 셀을 직접 실행합니다.

1. 환경 확인 → 2. 데이터 로드와 조회 → 3. 제외 피처 복구 → 4. 실험 설정 → 5. Dev 비교와 리포트 → 6. 고정 후보 Holdout.

- 운영 모델 `data/ml/models.joblib`, 운영 metadata, DB를 변경하지 않습니다.
- 결과는 `data/ml/experiments/<실행ID>/`에 HTML·Markdown·CSV·JSON, 선택 PDF로 생성됩니다.
- 기준모델 3개(최근 성장 지속·train 평균·0)와 HGB/Ridge/ElasticNet/SGD/RF/ExtraTrees/LightGBM/XGBoost를 비교할 수 있습니다.
- 현재 정답은 **다음 12개월 수출의 로그 성장**입니다. 6개월 실험은 6개월 label을 가진 새 피처 테이블이 필요합니다.
- 기존에 이미 본 2025-09 결과는 엄밀한 미사용 Holdout이 아닙니다. 여기의 예약은 새 비교에서의 역할 분리이며 과거 관측 이력을 없애지 않습니다.


## 1 환경 준비

이 PC 기본 Python(3.11.9, numpy 1.24.3, pandas 2.2.3, scikit-learn 1.9.1)으로 실행합니다. 저장된 운영 모델도 같은 환경에서 만들었습니다.
노트북에서 자동 설치는 하지 않습니다. 터미널에서 한 번만 실행하세요(기존 버전이 바뀌지 않게 고정).

```powershell
python -m pip install jupyterlab "numpy==1.24.3" "pandas==2.2.3" "scikit-learn==1.9.1"
# LightGBM/XGBoost는 쓸 때만
python -m pip install lightgbm xgboost "numpy==1.24.3" "pandas==2.2.3" "scikit-learn==1.9.1"
python -m jupyterlab notebooks/ml_model_a_experiments.ipynb
```

피처 스냅샷은 만든 환경과 같은 numpy·pandas 주 버전에서만 읽힙니다.

- `data/ml/features_snapshot_py311.pkl`: 이 PC Python 3.11로 만든 스냅샷 (12개월·6개월 정답 포함). **이 PC에서는 이 파일을 쓰세요.**
- `data/ml/features_snapshot.pkl`: numpy 2·pandas 3 환경에서 만든 보고서용 스냅샷. 이 PC 기본 Python으로는 읽히지 않습니다.
- 스냅샷이 없거나 원본부터 다시 만들려면 `DATA_SOURCE="raw"` (수 분 걸리고 메모리를 많이 씀).

새 실험은 저장 모델을 불러오지 않고 현재 커널에서 새로 학습합니다. pickle은 신뢰하는 로컬 파일만 여세요.


In [ ]:
from pathlib import Path
import sys, json
from dataclasses import replace

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "backend/app/ml/features.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("저장소 루트 또는 notebooks 폴더에서 실행하세요")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
from IPython.display import display, FileLink, HTML
from backend.app.ml import experiments as E
from backend.app.ml import feature_recovery as R
from backend.app.ml.experiment_report import write_reports
from backend.app.ml.runtime import runtime_versions

display(pd.Series(runtime_versions(), name="현재 커널"))
print("Python:", sys.executable)
print("Repository:", ROOT)


## 2 실행 스위치와 데이터 경로

처음에는 `LOAD_DATA=True`만 바꿔 피처·정답·결측을 확인하세요. 그 뒤 `RUN_DEV=True`로 바꿉니다.
피처 복구·민감도·분할 내보내기·Holdout·리포트 재생성은 각각 따로 켜는 스위치입니다.
스냅샷은 약 722MB이고 시나리오(original·recovered·민감도)는 원본을 공유하고 바뀐 열만 따로 들고 있습니다. 입력 행이 키(기준월·국가·HS6) 순서가 아닐 때만 실행 중 전체 사본이 하나 더 생깁니다.
메모리가 8GB인 PC에서는 시나리오를 한두 개씩 나눠 실행하세요.


In [ ]:
LOAD_DATA = False
DATA_SOURCE = "snapshot"     # "raw": 원본 전체 재전처리. 환율 조회만 DB 읽기 발생
RECOVER_EXTERNAL = False
RECOVER_FX = False
SENSITIVITY_RELEASE_ESTIMATE = False  # 현재 외부 값을 추정 공개일에 붙이는 민감도 시나리오 (Holdout·운영 금지)
EXPORT_SPLITS = False
RUN_DEV = False
RUN_HOLDOUT = False
REGENERATE_REPORT = False

SNAPSHOT = ROOT / "data/ml/features_snapshot_py311.pkl"  # 이 PC Python 3.11용. numpy 2 환경이면 features_snapshot.pkl
EXTERNAL_DIR = ROOT / "data/external"
# 과거 버전의 전처리 표가 있으면 경로 지정. None이면 현재 원본+availability.csv 사용.
MARKET_HISTORY = None
WDI_HISTORY = None
FX_MONTHLY = ROOT / "data/ml/recovery/fx_monthly.csv"
AVAILABILITY_ESTIMATE = ROOT / "backend/app/ml/reference/experiments/availability_release_estimate.csv"
COUNTRY_CURRENCY = None      # None: 국가 SQL 시드. 과거 통화 소속을 보장하는 표는 아님


In [ ]:
df = None
SCENARIOS = {}
if LOAD_DATA:
    if DATA_SOURCE == "snapshot":
        df = E.load_snapshot(SNAPSHOT)
    elif DATA_SOURCE == "raw":
        from backend.app.ml.artifacts import load_training_data
        df, raw_data, input_info = load_training_data()
        df.attrs["rebuilt_input"] = input_info
        del raw_data
    else:
        raise ValueError("DATA_SOURCE는 snapshot 또는 raw")
    SCENARIOS["original"] = df
    print("shape:", df.shape)
    display(df[["anchor", "country_code", "hs6", "exp_12m", "growth_12m", "y_growth"]].head(20))
    display(R.coverage(df))
else:
    print("데이터를 읽지 않았습니다. LOAD_DATA=True로 변경 후 이 셀을 실행하세요.")


## 3 제외된 11개 피처를 되살리는 방법

저장 모델은 45개 중 34개만 썼습니다. 빠진 11개는 값이 아예 없어서가 아니라 이유가 셋으로 나뉩니다.

|피처|빠진 이유|되살리는 방법 (권장 순)|이 노트북 스위치|
|---|---|---|---|
|환율 2개 (`fx_change_12m`, `fx_vol_12m`)|DB `fx_daily`가 0건|① ECOS 과거 적재 후 `DATA_SOURCE="raw"`로 피처 재생성 ② 근거 있는 월평균 CSV로 직접 연결|`DATA_SOURCE`, `RECOVER_FX`|
|WDI 4개 (`log_gdp`, `gdp_growth`, `log_population`, `log_gdp_pc`)|공개일표가 전부 2026-10-22(파일을 받은 날)이라 모든 기준월(~2026-09)보다 늦음|① WDI 과거 판(Database Archives)을 받아 `WDI_HISTORY` ② 민감도: 현재 값을 추정 공개일에 연결|`RECOVER_EXTERNAL`, `SENSITIVITY_RELEASE_ESTIMATE`|
|시장 5개 (`mkt_log_import`, `mkt_growth`, `kor_share`, `kor_share_change`, `mkt_hs4_growth`)|위와 같음. 게다가 BACI를 2022년부터 받아서 2022년 성장률을 만들 수 없음|① 과거 BACI 판과 공개일 근거로 `MARKET_HISTORY` ② BACI 수집 연도를 2019~2021까지 넓혀 성장률 이력 확보 ③ 민감도|같음|

환율 과거 적재 명령 (저장소 루트, `.env`에 ECOS 키와 DB 접속 정보 필요. 12개월 변화율에 13개월이 필요해 2022-01부터 받습니다):

```powershell
python -m backend.app.scheduler.jobs.fx_daily_load --start 2022-01-01 --end 2026-09-30
```

- **엄격 복구(①)**: 그 기준월 말에 실제로 공개돼 있던 판의 값만 붙입니다. Holdout·운영에 쓸 수 있는 유일한 방법입니다. 템플릿은 `backend/app/ml/reference/experiments/`의 `*_history_template.csv`, `fx_monthly_template.csv`입니다.
- **민감도**: `availability_release_estimate.csv`(공개 주기로 추정한 날짜, 모두 '확인 필요')로 지금 파일의 값을 연결합니다. 나중에 수정된 값을 미리 쓰는 셈이라 **낙관적**입니다. "과거 판을 구해 올 가치가 있나"를 가늠하는 데만 쓰고, 코드가 Holdout에서 이 시나리오를 거부합니다.
- 민감도에서 예상되는 모습: WDI는 거의 모든 기준월에 값이 붙습니다. 시장은 BACI 2022년 값이 2024-02 이후 기준월부터 붙지만, 성장률 3개는 2023년 값부터라 2025-02 이후에만 생깁니다. 그래서 12개월 학습 구간(~2024-09)에는 시장 성장률이 비어 학습에서 다시 빠질 수 있습니다. 정확한 관측률은 아래 셀의 표로 확인하세요.
- 환율은 사후 수정이 거의 없고 월말에 확정되므로 과거 적재만 하면 깨끗하게 살아납니다. 다만 국가 시드에 통화 코드가 없는 나라는 계속 비어 있습니다.
- `anomaly_ratio`는 사용 피처로 잡혀 있지만 R1 보정 미적용으로 항상 0입니다. 실험 설정 `drop_constant=True`이면 상수 열로 빠집니다. R1 임계값이 정해져야 살아납니다.
- 현재 `availability.csv`를 과거 날짜로 고쳐 쓰지 않습니다. 현재 외부 파일만으로 엄격 복구를 실행하면 **복구 값 0개**가 정상입니다.


In [ ]:
if RECOVER_EXTERNAL or RECOVER_FX:
    if df is None:
        raise RuntimeError("먼저 데이터 로드 셀을 실행하세요")
    recovered = df
    if RECOVER_EXTERNAL:
        recovered, external_audit = R.recover_external(
            recovered, external_dir=EXTERNAL_DIR,
            market_history=MARKET_HISTORY, wdi_history=WDI_HISTORY)
        display(external_audit)
    if RECOVER_FX:
        recovered, fx_audit = R.recover_fx(recovered, FX_MONTHLY, COUNTRY_CURRENCY)
        display(fx_audit)
    SCENARIOS["recovered"] = recovered
    print("원본과 복구 자료의 동일 행·정답 조건은 실험 실행 전에 다시 확인합니다.")
else:
    print("피처 복구 미실행. original 시나리오로 비교할 수 있습니다.")

if SENSITIVITY_RELEASE_ESTIMATE:
    if df is None:
        raise RuntimeError("먼저 데이터 로드 셀을 실행하세요")
    sensitivity, sensitivity_audit = R.recover_external(df, external_dir=EXTERNAL_DIR, availability=AVAILABILITY_ESTIMATE)
    display(sensitivity_audit)
    SCENARIOS["sensitivity_release_estimate"] = sensitivity
    print("민감도 시나리오: 수정값을 추정 공개일에 미리 쓴 낙관적 비교용입니다. Holdout·운영에 쓰지 않습니다.")


## 4 모델과 실험 조건

기본은 HGB·Ridge·ExtraTrees, 피처 조합 2개, Dev 기준월 2개입니다. 기준선 3개가 각 조합에 자동 포함됩니다.
초기 작동/시간 확인은 `max_train_rows=100_000`, 정식 비교는 `None`으로 정하세요. 샘플링 여부는 리포트에 남습니다.
평가(test)는 샘플링하지 않아 추천 후보 집합을 유지합니다. 같은 train 샘플을 모든 모델·복구 시나리오에 적용합니다.
이 설정으로 실행하면 기본 학습 모델 12회와 기준선 12회를 평가합니다. 모델 내부 CPU는 n_jobs로 제한합니다.


In [ ]:
MODELS = [
    E.ModelSpec("hgb_300", "hgb", {"max_iter": 300, "learning_rate": 0.05}),
    E.ModelSpec("ridge_10", "ridge", {"alpha": 10.0}),
    E.ModelSpec("extra_trees_200", "extra_trees", {"n_estimators": 200, "max_depth": 14}),
    # E.ModelSpec("elasticnet", "elasticnet", {"alpha": 0.001, "l1_ratio": 0.2}),
    # E.ModelSpec("sgd_huber", "sgd", {"loss": "huber"}),
    # E.ModelSpec("random_forest", "random_forest", {"n_estimators": 200}),
    # E.ModelSpec("lightgbm_300", "lightgbm", {"n_estimators": 300, "num_leaves": 31}),
    # E.ModelSpec("xgboost_300", "xgboost", {"n_estimators": 300, "max_depth": 6}),
]
FEATURE_SETS = {
    "all": E.FEATURE_SETS["all"],
    "without_related": E.FEATURE_SETS["without_related"],
    # "without_news": E.FEATURE_SETS["without_news"],
    # "core": E.FEATURE_SETS["core"],
    # "without_external_fx": E.FEATURE_SETS["without_external_fx"],
}
CONFIG = E.ExperimentConfig(
    validation_anchors=("2025-02-01", "2025-05-01"),
    holdout_anchor="2025-09-01", phase="dev",
    horizon=12, gap_months=12,
    ks=(5, 10), min_candidates=10,
    seeds=(0,), n_jobs=2,
    max_train_rows=None, sampling_seed=0,
    target_clip=(0.01, 0.99), drop_constant=True,
    direction_threshold=0.0, primary_metric="country_ndcg_at_5",
    save_predictions=True, save_models=False, make_pdf=True,
    output_root=str(ROOT / "data/ml/experiments"),
)
display(E.dependency_status(MODELS))
print("피처 조합:", {k: len(v) for k, v in FEATURE_SETS.items()})
print("자료 시나리오:", list(SCENARIOS))
print("총 평가 건수:", len(SCENARIOS)*len(FEATURE_SETS)*len(CONFIG.validation_anchors)*len(CONFIG.seeds)*(len(MODELS)+3))


### 추가 확인 예시와 모델별 특징

아래 코드는 Markdown 예시라 자동 실행되지 않습니다. 필요한 줄만 위 설정 셀에 옮겨 쓰세요.

```python
# 작은 grid도 모델 수 × fold × 피처 묶음 × 시나리오 × seed만큼 실행됩니다.
from sklearn.model_selection import ParameterGrid
MODELS = [E.ModelSpec(f"hgb_grid_{i}", "hgb", p)
          for i, p in enumerate(ParameterGrid({"max_leaf_nodes": [15, 31], "learning_rate": [0.03, 0.05], "max_iter": [300]}))]
CONFIG = replace(CONFIG, seeds=(0, 42), target_clip=None)  # 정답 자르기 없이 비교
# 6개월: DATA_SOURCE="raw"로 새 스냅샷을 만든 뒤 horizon=6, gap_months=6 (기준선은 자동으로 단순 지속 규칙).
# 기존 12개월 피처 스냅샷을 6개월 정답이라고 바꿔 쓰지 않습니다.
```

|모델 (kind)|성격|전처리|처음 돌릴 때|
|---|---|---|---|
|`hgb`|운영 모델과 같은 계열의 부스팅|결측 그대로, 범주는 정수 코드(native categorical). 내부 early stopping 끔|기준 후보|
|`lightgbm`, `xgboost`|다른 구현의 부스팅 (선택 설치)|결측 그대로, 범주는 정수 코드. XGBoost도 원핫·밀집 행렬을 쓰지 않아 메모리를 적게 씀|미설치면 skipped로 기록|
|`random_forest`, `extra_trees`|배깅 트리|중앙값 대체·결측 표시·원핫(희소)|학습 행이 많으면 오래 걸림. 먼저 `max_train_rows=100_000`으로 시간을 재 보세요|
|`ridge`, `elasticnet`, `sgd`|선형 모델 (비선형 못 잡는 하한 비교용)|중앙값 대체·결측 표시·표준화·원핫|빠름|

기준선 3개(작년 성장률 또는 6개월 단순 지속, train 평균, 0)는 모든 조합에 자동으로 들어갑니다.
피처 자르기·결측 대체·인코딩·스케일·정답 자르기 경계는 모두 각 분할의 train에서만 계산합니다. 평가 행으로 반복 수를 고르지 않습니다.


## 5 실제 train/test와 피처 관측률 확인

이 셀은 모델 학습을 하지 않습니다. 선택한 시나리오의 첫 Dev 분할을 보여 줍니다.
`y_fit`은 학습 때만 자르는 정답이며 평가에는 자르기 전 `y_growth`를 사용합니다.


In [ ]:
PREVIEW_SCENARIO = "original"
PREVIEW_FEATURE_SET = "all"
train = test = None
if SCENARIOS:
    preview = E.prepare_scenarios({PREVIEW_SCENARIO: SCENARIOS[PREVIEW_SCENARIO]}, CONFIG.horizon)[PREVIEW_SCENARIO]
    train, test = E.split_rows(preview, CONFIG.validation_anchors[0], CONFIG)
    columns = FEATURE_SETS[PREVIEW_FEATURE_SET]
    display(R.coverage(train, columns).merge(R.coverage(test, columns), on="feature", suffixes=("_train", "_test")))
    print("train", train.shape, train.anchor.min(), train.anchor.max())
    print("test", test.shape, test.anchor.min(), test.anchor.max())
    display(train[["anchor", "country_code", "hs6", *columns, "y_growth"]].head(20))
    display(test[["anchor", "country_code", "hs6", *columns, "y_growth"]].head(20))
    del preview
else:
    print("데이터 로드 후 확인할 수 있습니다.")


In [ ]:
if EXPORT_SPLITS:
    if train is None:
        raise RuntimeError("먼저 train/test 확인 셀을 실행하세요")
    from datetime import datetime
    from zoneinfo import ZoneInfo
    from uuid import uuid4
    export_dir = ROOT / "data/ml/experiments" / ("split_preview_" + datetime.now(ZoneInfo("Asia/Seoul")).strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:8])
    export_dir.mkdir(parents=True, exist_ok=False)
    selected = R.coverage(train, FEATURE_SETS[PREVIEW_FEATURE_SET])
    usable = selected[selected.status == "available" if CONFIG.drop_constant else selected.status != "all_missing"].feature.tolist()
    bounds = train.y_growth.quantile(CONFIG.target_clip).tolist() if CONFIG.target_clip else None
    for name, rows in [("train", train), ("test", test)]:
        out = rows[["anchor", "country_code", "hs6", *usable, "y_growth"]].copy()
        if name == "train":
            out["y_fit"] = out.y_growth.clip(*bounds) if bounds else out.y_growth
        out.to_csv(export_dir / f"{name}.csv.gz", index=False)
        out.head(100).to_csv(export_dir / f"{name}_preview100.csv", index=False, encoding="utf-8-sig")
    print("저장:", export_dir)
    print("모델별 인코딩/스케일 전의 공통 입력입니다. test label은 절단하지 않습니다.")
else:
    print("분할 파일 내보내기 미실행")


## 6 Dev 모델 비교 실행

**이 셀이 실제 fit/predict를 수행합니다.** `RUN_DEV=True`로 바꾼 경우에만 실행됩니다.
모델은 순차 실행하고 현재 커널에서 진행 상황을 출력합니다. 실패·미설치 모델도 리포트에 기록합니다.
중단하면 완료된 결과까지 보존하지만 중간 결과를 완전한 비교 후보로 올리지 않습니다.


In [ ]:
DEV_RUN = None
if RUN_DEV:
    if not SCENARIOS:
        raise RuntimeError("먼저 데이터 로드와 원하는 복구 셀을 실행하세요")
    DEV_RUN = E.run_experiments(SCENARIOS, MODELS, FEATURE_SETS, CONFIG)
    print("완료:", DEV_RUN)
    display(E.leaderboard(DEV_RUN))
    display(FileLink(str(DEV_RUN / "report.html")))
    if (DEV_RUN / "report.pdf").exists():
        display(FileLink(str(DEV_RUN / "report.pdf")))
else:
    print("모델 학습·평가를 실행하지 않았습니다. RUN_DEV=True로 변경 후 이 셀을 실행하세요.")


## 7 리포트 읽기와 재생성

- `report.html`: 비교 차트, 요약표, **지표별 비교표**(지표 × 후보, '최고 후보' 표시), 실행별 전체 지표, 품목군별 결과, 지표 정의, 설정·데이터 계보. 브라우저 인쇄로 PDF 저장도 됩니다.
- `report.pdf`: `make_pdf=True`일 때 생성. 지표별 비교표는 후보 6개씩 나눠 싣습니다. 한글 폰트 실패는 `pdf_status.json`에 남고 HTML은 보존됩니다.
- `metric_matrix.csv`: 지표별 비교표 원본.
  - 순위: 국가별·품목별 Top K의 Precision, NDCG, Hit, 상위 20% Precision·Recall·F1, 추천 중 실제 증가 비율, 추천의 실제 성장 평균, 그룹 내 Spearman
  - 증가 여부: Accuracy, Balanced accuracy, Precision, Recall, F1, MCC, ROC-AUC, Average precision
  - 성장률 회귀: MAE, RMSE, Median AE, R², Explained variance, Pearson, Spearman, 학습 시간
- `leaderboard.csv`: 모든 fold/seed의 평균·표준편차·유효 실행 수. 빠진 fold가 있으면 complete=False.
- `fold_metrics.csv`: 실행 하나하나의 전체 지표와 혼동행렬(tn/fp/fn/tp).
- `segments.csv`: 반도체와 그 외 품목군. `__groups.csv`: 국가별·품목별 그룹 점수.
- `__coverage.csv`: 분할별 피처 관측률. `__predictions.csv.gz`: 평가 행 예측. `__train_keys.csv.gz`: 학습에 쓴 행 키.
- `manifest.json`, `results.json`: 설정·환경·입력 해시·실패·시간·파라미터·사용 피처 기록.

NDCG는 정확도가 아닙니다. AP는 average precision이며 사다리꼴 PR-AUC와 다릅니다.
점수는 확률이 아니므로 log-loss/Brier는 계산하지 않습니다. 미평가 지표는 0이 아니라 null(표에서는 '미평가')입니다.


In [ ]:
# 이미 실행된 폴더를 확인하려면 아래 경로를 지정하세요. 재학습은 하지 않습니다.
EXISTING_RUN = None  # ROOT / "data/ml/experiments/실행ID"
REPORT_RUN = Path(EXISTING_RUN) if EXISTING_RUN else DEV_RUN
if REPORT_RUN is not None:
    board = E.leaderboard(REPORT_RUN)
    display(board)
    manifest = json.loads((REPORT_RUN / "manifest.json").read_text("utf-8"))
    print("status:", manifest["status"], "phase:", manifest["config"]["phase"])
    if REGENERATE_REPORT:
        write_reports(REPORT_RUN, make_pdf=True)
    display(FileLink(str(REPORT_RUN / "report.html")))
else:
    print("결과 폴더를 지정하거나 Dev 실험을 먼저 직접 실행하세요.")


## 8 후보를 고정한 뒤 Holdout 한 번 확인

Dev 리포트에서 모델·피처·자료 시나리오를 정한 뒤 아래 값을 바꿉니다.
모든 Dev fold가 완료되고 주 지표가 계산된 후보만 설정을 복사할 수 있습니다. 모델 파라미터·샘플링·label 절단은 Dev와 같습니다.
이 셀도 `RUN_HOLDOUT=False`가 기본입니다. Holdout을 본 뒤 다시 설정을 고르면 그 시점은 이후 Dev로 취급해야 합니다.


In [ ]:
SELECTED_SCENARIO = "original"
SELECTED_FEATURE_SET = "all"
SELECTED_MODEL = "hgb_300"
SELECTED_SEED = 0
HOLDOUT_RUN = None
if RUN_HOLDOUT:
    if REPORT_RUN is None or not SCENARIOS:
        raise RuntimeError("Dev 결과 폴더와 동일 자료 시나리오가 필요합니다")
    holdout_config, selected_models, selected_features = E.holdout_configuration(
        REPORT_RUN, SELECTED_SCENARIO, SELECTED_FEATURE_SET, SELECTED_MODEL, SELECTED_SEED)
    HOLDOUT_RUN = E.run_experiments(
        {SELECTED_SCENARIO: SCENARIOS[SELECTED_SCENARIO]}, selected_models, selected_features, holdout_config)
    print("Holdout:", HOLDOUT_RUN)
    display(E.leaderboard(HOLDOUT_RUN))
    display(FileLink(str(HOLDOUT_RUN / "report.html")))
else:
    print("Holdout 학습·평가 미실행")


## 9 결과 해석과 남은 확인

1. 복구 뒤 train에서도 값이 생겼는지 확인합니다. test만 값이 생긴 피처는 학습 구간에서 제거됩니다.
2. 피처 유무가 같은 조합은 사실상 동일 실험입니다. 외부 통계가 전부 결측이면 all과 without_external_fx가 같은 입력일 수 있습니다.
3. 기본 HGB는 내부 무작위 early stopping을 끄고 상수 피처도 제외하므로 기존 보고서 0.7106의 완전 동일 재현 실험이 아닙니다.
4. 모델마다 인코딩이 다릅니다. 공통은 train/test·정답·후보·seed·피처 선택이며 각 모델에 맞는 전처리도 비교 구성의 일부입니다.
5. train 행 샘플링은 계산량을 줄이는 실험 옵션입니다. 전체 학습 성능으로 표현하지 않습니다.
6. 단신속보 게시일·현재 태그, 관세청 통계의 공표 지연·수정본 문제는 기존 입력에 남아 있습니다. 순위 성능과 point-in-time 데이터 완전성은 별개입니다.
7. 모델 비교에서 좋은 결과가 나와도 운영 모델로 자동 승격하지 않습니다. 별도 검토 후 결정합니다.

자세한 복구 조건과 실행 방법: `docs/ML_A_EXPERIMENTS.md`.

공식 API 근거:
- https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html
- https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html
- https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.LGBMRegressor.html
- https://xgboost.readthedocs.io/en/stable/python/python_api.html
